In [1]:
!pip install openai faiss-cpu

In [2]:
import openai, time, uuid
import faiss
import numpy as np
from typing import List, Dict

In [3]:
resume_contexts = [
    "Engineering Manager & Technical Architect (AffinityX)",
    "InDesign developer, scripting & automation",
    "Enterprise AI/MLOps orchestration (AWS Bedrock, Lambda, Step Functions)",
    "Python notebook workflow design",
    "OpenAI LLM integration with monitoring & error handling"
]

In [4]:
def build_index(texts: List[str]):
    embeddings = [openai.Embedding.create(model="text-embedding-3-small", input=t)["data"][0]["embedding"] for t in texts]
    dim = len(embeddings[0])
    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings).astype("float32"))
    return index, embeddings

index, resume_embeddings = build_index(resume_contexts)

def retrieve_context(query: str, k: int = 2) -> List[str]:
    q_emb = openai.Embedding.create(model="text-embedding-3-small", input=query)["data"][0]["embedding"]
    D, I = index.search(np.array([q_emb]).astype("float32"), k)
    return [resume_contexts[i] for i in I[0]]

In [5]:
def monitor_latency(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        latency = time.time() - start
        print(f"[MONITOR] Latency: {latency:.2f}s")
        return result
    return wrapper

def log_event(event_type: str, details: Dict):
    cid = str(uuid.uuid4())
    print(f"[LOG] {event_type} | CID={cid} | Details={details}")
    return cid

In [6]:
@monitor_latency
def generate_answer(question: str, retries: int = 3) -> str:
    context = retrieve_context(question)
    prompt = f"""
    You are a candidate in a technical interview.
    Question: {question}
    Use the following resume context to craft a professional answer:
    {context}
    """

    for attempt in range(retries):
        try:
            response = openai.ChatCompletion.create(
                model="gpt-4-turbo",
                messages=[{"role": "system", "content": "You are a helpful interview assistant."},
                          {"role": "user", "content": prompt}],
                max_tokens=300,
                temperature=0.7
            )
            answer = response.choices[0].message["content"]
            log_event("AnswerGenerated", {"question": question, "attempt": attempt+1})
            return answer
        except Exception as e:
            log_event("Error", {"error": str(e), "attempt": attempt+1})
            if attempt == retries - 1:
                return "Error: Unable to generate answer after retries."

In [7]:
questions = [
    "Can you describe your experience integrating OpenAI models into enterprise workflows?",
    "How do you ensure scalability and performance in Python-based AI solutions?",
    "Tell me about a project where you applied LLMs in notebooks for business use cases."
]

qa_sheet = []
for q in questions:
    ans = generate_answer(q)
    qa_sheet.append({"Question": q, "Answer": ans})

[LOG] AnswerGenerated | CID=f781211c-2928-4aa8-b461-551197aef684 | Details={'question': 'Can you describe your experience integrating OpenAI models into enterprise workflows?', 'attempt': 1}
[MONITOR] Latency: 9.75s
[LOG] AnswerGenerated | CID=49ecb618-e652-4509-86a5-c0fc19992ffb | Details={'question': 'How do you ensure scalability and performance in Python-based AI solutions?', 'attempt': 1}
[MONITOR] Latency: 11.44s
[LOG] AnswerGenerated | CID=c02dfe25-6907-41ca-84f0-c47d50536a7d | Details={'question': 'Tell me about a project where you applied LLMs in notebooks for business use cases.', 'attempt': 1}
[MONITOR] Latency: 9.68s


In [8]:
for item in qa_sheet:
    print("Q:", item["Question"])
    print("A:", item["Answer"])
    print("-" * 50)

Q: Can you describe your experience integrating OpenAI models into enterprise workflows?
A: Certainly! My experience with integrating OpenAI models into enterprise workflows is extensive and has primarily focused on enhancing business processes through automation and intelligent decision-making.

One of the key projects involved integrating OpenAI's large language models (LLMs) into an existing business workflow. My role was to ensure that the integration was seamless and efficient. This involved setting up robust monitoring and error-handling frameworks to ensure the system's reliability and accuracy. For monitoring, I implemented logging and alerting mechanisms that would notify the team of any performance anomalies or failures in real-time. Error handling was meticulously designed to manage and mitigate any interruptions in the workflow, ensuring that fallback mechanisms were in place to maintain operational continuity.

In addition to the direct OpenAI LLM integration, I was also i